[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wildertrek/catcher-in-the-cache/blob/main/notebooks/06_register_matched_synth.ipynb)

# 06: Register-Matched Substrate Falsifier (robustness control)

**Research question.** Does the canonical->synthetic H<->A_HEX collapse survive
*register-matching*? The 20 original synthetic characters are modern-register
prose, so a reviewer could argue the collapse reflects a prose-register shift
rather than absence from the training corpus. We test it directly: generate
**period-matched (19th-century) synthetic characters** under the same
H<->A_HEX decoupling, then re-rate canonical + modern-synth + register-synth
with the **same** raters.

**Result (preview).** Within-rater r(H, A_HEX): canonical **+0.65** ->
modern-synth **-0.13**, register-matched **+0.02**. The collapse is
register-independent -> driven by out-of-corpus-ness. Reproduces at $0 from
cached artifacts.

## Setup

**In Colab, run the cell below first.** It clones the companion repository and installs dependencies (~30 s), then changes into `notebooks/` so the relative paths in this notebook resolve. Run locally, the cell is a no-op.

In [ ]:
# Colab setup: clone the companion repo + install dependencies (no-op when run locally).
import sys, os, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("catcher-in-the-cache"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Wildertrek/catcher-in-the-cache.git"], check=True)
    os.chdir("catcher-in-the-cache/notebooks")
    subprocess.run(["pip", "install", "-q", "-r", "../requirements.txt"], check=True)
    print("Colab setup complete. Working directory:", os.getcwd())

## Method
- **Generate** 10 new synthetic characters in 19th-c literary register
  (Austen / Eliot / Dickens diction, period settings, no modern reference),
  same author-designed H<->A_HEX decoupling (designed r approx -0.70), with the
  3-provider novelty negative-control. Script: `pivot6_generate_register_matched.py`.
- **Rate** register-matched (10) + modern-synth (20) + a 15-char canonical
  sample with the SAME 3 current frontier raters (Opus 4.6, GPT-4o, Gemini 3.1
  Pro), names redacted, using the exact panel HEXACO probe. Scripts:
  `pivot6_rate_register_matched.py`, `pivot6_rate_canonical_baseline.py`.
- **Compare** within-rater r(H, A_HEX) per substrate (same-rater, signed).

In [1]:
import json, statistics
from pathlib import Path
ART = Path("../paper_artifacts/pivot6_hexaco_atlas/register_matched")
if not ART.exists():
    ART = Path("paper_artifacts/pivot6_hexaco_atlas/register_matched")

# Design check + novelty control
sub = json.loads((ART / "register_matched_substrate.json").read_text())
chars = sub["characters"]
H = [c["target_HEXACO"]["H"] for c in chars]
A = [c["target_HEXACO"]["A_HEX"] for c in chars]
ver = json.loads((ART / "register_matched_verification.json").read_text())
print(f"register-matched characters : {len(chars)}")
print(f"designed r(H, A_HEX)        : {statistics.correlation(H, A):+.3f}  (target: strongly negative)")
print(f"novelty control flagged     : {len(ver['flagged_indices'])}/{len(chars)}  (0 = all out-of-corpus)")
print(f"sample names                : {[c['op_name'] for c in chars[:5]]}")

register-matched characters : 10
designed r(H, A_HEX)        : -0.704  (target: strongly negative)
novelty control flagged     : 0/10  (0 = all out-of-corpus)
sample names                : ['Mr. Cedric Pallingham', 'Mrs. Lavinia Doole', 'Lieutenant Everard Stannard', 'Miss Perpetua Blackall', 'Mr. Josias Ruck']


In [2]:
# Same-rater three-way comparison (within-rater r(H, A_HEX))
res = json.loads((ART / "rating_results.json").read_text())
t = res["three_way_same_rater"]
print(t["note"], "\n")
print(f"{'substrate':20s} {'Opus4.6':>9s} {'GPT-4o':>9s} {'Gemini3.1':>10s} | {'mean r':>8s}")
for s in ["canonical_sample", "modern_synth", "register_matched"]:
    p = t[s]["per_rater_r"]
    print(f"{s:20s} {p['anthropic_opus46']:>9} {p['openai_gpt4o']:>9} "
          f"{p['google_gemini31']:>10} | {t[s]['mean_r']:>8}")

all three substrates rated by the SAME 3 raters (Opus 4.6, GPT-4o, Gemini 3.1 Pro) 

substrate              Opus4.6    GPT-4o  Gemini3.1 |   mean r
canonical_sample         0.658    0.8721     0.4249 |   0.6517
modern_synth            -0.301    0.3856    -0.4686 |   -0.128
register_matched       -0.1165    0.4813    -0.3189 |   0.0153


## Verdict

Within-rater r(H, A_HEX) drops from **+0.65 on canonical** characters to
**~0 on both synthetic substrates** (modern -0.13, period-matched +0.02).
Period-matched 19th-century prose does **not** restore the fusion, so the
substrate-falsifier collapse is driven by **absence from the training corpus,
not by prose register.** The register confound is ruled out.

**Caveats (audit -> test -> re-audit).**
- Focused **3-rater** replication, not the full 25-rater panel; framed as a
  robustness check.
- Small n (canonical 15, modern 20, register 10) -> wide intervals.
- The **signed** means above are the honest summary: a mean-|r| would mask that
  2 of 3 raters go *negative* on the synthetic sets (they partly recover the
  designed anti-correlation).
- The same-rater canonical baseline (+0.65) **replicates** the published
  25-rater panel value (+0.75), validating the comparison.